In [1]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib

In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2026-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
  open_df = df3["Open"]
  volume_df = df3["volume"]

except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)
  open_df = df3.xs("Open", level = 0, axis = 1)
  volume_df = df3.xs("Volume", level = 0, axis = 1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")
open_df = open_df.dropna(axis=1, how = "all")
volume_df = volume_df.dropna(axis =1 , how = "all")

valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns).intersection(open_df.columns).intersection(volume_df.columns)
)
close_df, high_df, low_df, open_df, volume_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
    open_df[valid_cols],
    volume_df[valid_cols]
)




[*********             19%                       ]  95 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************50%                       ]  248 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************60%****                   ]  301 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[*********************100%***********************]  500 of 500 completed

3 Failed downloads:
['VYLR', 'HONA', 'FDXF']: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")


### Stochastic oscillator reversal strategy

In [5]:
macd_dict = {}
signal_dict = {}
hist_dict = {}
rsi_dict = {}
stoch_k_dict = {}
stoch_d_dict = {}


for ticker in close_df.columns:
    prices = close_df[ticker].to_numpy(dtype=np.float64)
    high_series = high_df[ticker]
    low_series = low_df[ticker]
    volume_values = volume_df[ticker].values

    macd, signal, hist = talib.MACD(
        prices,
        fastperiod=12,
        slowperiod=26,
        signalperiod=9
    )

    rsi = talib.RSI(
        prices,
        timeperiod=14
    )

    stoch_k, stoch_d = talib.STOCH(
        high_series.values,
        low_series.values,
        prices,
        fastk_period=14,
        slowk_period=3,
        slowk_matype=0,
        slowd_period=3,
        slowd_matype=0
    )
    


    macd_dict[ticker] = macd
    signal_dict[ticker] = signal
    hist_dict[ticker] = hist
    rsi_dict[ticker] = rsi
    stoch_k_dict[ticker] = stoch_k
    stoch_d_dict[ticker] = stoch_d

   

macd_df = pd.DataFrame(macd_dict, index=close_df.index)
signal_df = pd.DataFrame(signal_dict, index=close_df.index)
hist_df = pd.DataFrame(hist_dict, index=close_df.index)
rsi_df = pd.DataFrame(rsi_dict, index = close_df.index )
stoch_k_df = pd.DataFrame(stoch_k_dict, index = close_df.index)
stoch_d_df = pd.DataFrame(stoch_d_dict, index = close_df.index)

entries = (
    (stoch_k_df > stoch_d_df)
    & (stoch_k_df.shift(1) <= stoch_d_df.shift(1))
    & (stoch_k_df < 20)
).shift(1).fillna(False).astype(bool)

exits = (
    (stoch_k_df < stoch_d_df)
    & (stoch_k_df.shift(1) >= stoch_d_df.shift(1))
    & (stoch_k_df > 80)
).shift(1).fillna(False).astype(bool)


backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

equity = backtest1.value(group_by=True)
equity.vbt.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               356402.005776
Total Return [%]                           256.402006
Benchmark Return [%]                       524.559277
Max Gross Exposure [%]                          100.0
Total Fees Paid                          45695.407395
Max Drawdown [%]                            36.451402
Max Drawdown Duration               530 days 00:00:00
Total Trades                                    20956
Total Closed Trades                             20799
Total Open Trades                                 157
Open Trade PnL                            7022.854182
Win Rate [%]                                62.137603
Best Trade [%]                             225.754665
Worst Trade [%]                             -45.45175
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### State based stochastic oscillator implementation

In [9]:
entries = ((stoch_k_df < 20) & (stoch_k_df > stoch_d_df)).shift(1).fillna(False).astype(bool)

exits = ((stoch_k_df > 80) & (stoch_k_df < stoch_d_df)).shift(1).fillna(False).astype(bool)

backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()


Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               321464.805762
Total Return [%]                           221.464806
Benchmark Return [%]                       524.559277
Max Gross Exposure [%]                          100.0
Total Fees Paid                          41399.324646
Max Drawdown [%]                            36.773904
Max Drawdown Duration               432 days 00:00:00
Total Trades                                    21330
Total Closed Trades                             21170
Total Open Trades                                 160
Open Trade PnL                             6860.21029
Win Rate [%]                                61.955598
Best Trade [%]                             225.754665
Worst Trade [%]                             -45.45175
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### Stochastic oscillator with RSI confirmation

In [10]:
entries = (
    (stoch_k_df > stoch_d_df)
    & (stoch_k_df.shift(1) <= stoch_d_df.shift(1))
    & (stoch_k_df < 20)
    & (rsi_df < 30)
).shift(1).fillna(False).astype(bool)

exits = (
    (stoch_k_df < stoch_d_df)
    & (stoch_k_df.shift(1) >= stoch_d_df.shift(1))
    & (stoch_k_df > 80)
    & (rsi_df > 70)
).shift(1).fillna(False).astype(bool)

backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()


Start                              2015-05-05 00:00:00
End                                2026-05-04 00:00:00
Period                              2766 days 00:00:00
Start Value                                   100000.0
End Value                                310751.364463
Total Return [%]                            210.751364
Benchmark Return [%]                        524.559277
Max Gross Exposure [%]                           100.0
Total Fees Paid                           12921.819606
Max Drawdown [%]                             37.927309
Max Drawdown Duration                534 days 00:00:00
Total Trades                                      4007
Total Closed Trades                               3902
Total Open Trades                                  105
Open Trade PnL                            17571.935546
Win Rate [%]                                 39.390056
Best Trade [%]                              256.625901
Worst Trade [%]                             -35.981281
Avg Winnin

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data

